In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Import the image and save it as an array

In [ ]:
import os
from PIL import Image

def load_images_from_folder(folder):
    # Dictionary to store images and their labels (actually a list in this implementation)
    images_data = []

    # Iterate through all files in the folder
    for filename in os.listdir(folder):
        if filename.endswith(".tiff"):  # Ensure only TIFF files are processed
            # Split filename to extract name, label+number, and index
            name_parts = filename.split('.')
            name = name_parts[0]
            label_number = name_parts[1]
            index = name_parts[2]

            # Extract the label and the trailing digit
            label = label_number[:-1]    # All characters except the last one
            number = label_number[-1]    # The last character (the number)

            # Load the image
            image_path = os.path.join(folder, filename)
            with Image.open(image_path) as img:
                images_data.append({
                    'image': img.copy().convert('L'),  # Convert to grayscale
                    'name': name,
                    'label': label,
                    'number': number,
                    'index': index
                })
            img.close()  # Ensure the file is closed promptly

    return images_data

In [ ]:
# Use the function
folder_path = './data/jaffe'  # Replace with your actual image folder path
images = load_images_from_folder(folder_path)
for img_data in images:
    print(f"Name: {img_data['name']}, Label: {img_data['label']}, Number: {img_data['number']}, Index: {img_data['index']}")
    image = np.array(img_data['image'])
    plt.imshow(image, cmap='gray')  # Display the image in grayscale
    plt.colorbar()  # Optional: show colorbar
    plt.show()
    break  # Only display the first image and then exit the loop

In [ ]:
images[0]

In [ ]:
import pandas as pd

# File path
file_path = 'label.txt'  # Replace with your actual file path

# Read the data using pandas, skipping the first two rows
data = pd.read_csv(file_path, delim_whitespace=True, skiprows=2, header=None)

# Set column names
column_names = ['index', 'HAP', 'SAD', 'SUR', 'ANG', 'DIS', 'FEA', 'PIC']
data.columns = column_names

# Drop the last column 'PIC'
data.drop('PIC', axis=1, inplace=True)

# Set 'index' as the DataFrame index
data.set_index('index', inplace=True)

# Display the data to verify correct reading and processing
print(data.head())  # Print the first few rows

In [ ]:
# Create a new list to store the matched images and corresponding label data
images_data = []
label = []
counts = []

for image_info in images:
    img_index = int(image_info['index'])  # Ensure index is an integer
    if img_index in data.index:
        # Combine the DataFrame row (all emotion scores) with the image information
        images_data.append(np.array(image_info['image']))
        label.append(image_info['label'])
        counts.append(np.array(data.loc[img_index]))
    else:
        print(f"No data found for image with index {img_index}")

In [ ]:
data = np.array(images_data)
label = np.array(label)
counts = np.array(counts)

In [ ]:
# save data
directory = './data/jaffe/'
np.save(f'{directory}data_raw.npy', data)
np.save(f'{directory}label_raw.npy', label)
np.save(f'{directory}counts_raw.npy', counts)

### Align the mouth and eyes of the picture

In [ ]:
import cv2
import dlib
import os
import numpy as np

def align_and_crop_face(image, eye_left, eye_right, mouth_center, desired_size=100):
    # Define target positions for left eye, right eye, and mouth center in the output image
    eye_left_target = (int(desired_size * 0.3), int(desired_size * 0.4))
    eye_right_target = (int(desired_size * 0.7), int(desired_size * 0.4))
    mouth_center_target = (int(desired_size * 0.5), int(desired_size * 0.75))

    # Original feature points detected in the image
    original_points = np.float32([eye_left, eye_right, mouth_center])
    # Desired positions in the aligned image
    target_points = np.float32([eye_left_target, eye_right_target, mouth_center_target])

    # Compute affine transformation matrix
    M = cv2.getAffineTransform(original_points[:3], target_points[:3])
    # Apply affine warp to align the face
    aligned_image = cv2.warpAffine(image, M, (desired_size, desired_size))
    
    return aligned_image

# Initialize face detector and landmark predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('./data/shape_predictor_68_face_landmarks.dat')

# Set input and output directories
input_directory = './data/jaffe/'
output_directory = './data/pre_jaffe/'

# Create output directory if it does not exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Iterate through each file in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith(".tiff"):  # Process only image files
        input_path = os.path.join(input_directory, filename)
        image = cv2.imread(input_path)
        if image is None:
            continue
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Detect faces in the grayscale image
        faces = detector(gray, 1)
        for face in faces:
            landmarks = predictor(gray, face)

            # Calculate coordinates of left eye, right eye, and mouth centers
            eye_left = ((landmarks.part(36).x + landmarks.part(39).x) // 2, (landmarks.part(36).y + landmarks.part(39).y) // 2)
            eye_right = ((landmarks.part(42).x + landmarks.part(45).x) // 2, (landmarks.part(42).y + landmarks.part(45).y) // 2)
            mouth_center = ((landmarks.part(48).x + landmarks.part(54).x) // 2, (landmarks.part(48).y + landmarks.part(54).y) // 2)

            # Align and crop the face
            aligned_image = align_and_crop_face(image, eye_left, eye_right, mouth_center)

            # Build output file path
            output_path = os.path.join(output_directory, filename)

            # Save the aligned and cropped image
            cv2.imwrite(output_path, aligned_image)

print("All images have been processed and saved.")

### Data import and processing

In [ ]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import dcor
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from support_function import *

In [ ]:
directory = './data/jaffe/'
data = np.load(f'{directory}data.npy')
label = np.load(f'{directory}label.npy')
counts = np.load(f'{directory}counts.npy')

In [ ]:
wire_data = data.reshape(213, -1)/255.0
data = data[:, np.newaxis, :, :]/255.0

In [ ]:
dis_sum = counts.sum(axis=1)[:,np.newaxis]
dis = np.tile(dis_sum, (1,6))
distribution = counts/dis

In [ ]:
print("Min:", data.min())
print("Max:", data.max())

In [ ]:
emotion = {'HA':0, 'SA':1, 'SU':2, 'AN':3, 'DI':4, 'FE':5, 'NE':6}

def get_label(emo, label, distribution, mode=0):
    n = label.shape[0]
    if mode == 0:  # Compute single (hard) label
        single_label = np.zeros(n)
        for i in range(n):
            single_label[i] = emo[label[i]]
        return single_label.astype(np.int8)
    elif mode == 1:   # Compute multi-label (soft label)
        multi_label = np.zeros((n, 7))
        r = 0.2       # Set threshold to 0.2
        for i, dis in enumerate(distribution):
            multi_label[i, :6] = (dis > r).astype(np.int8)   # Apply threshold to first 6 emotions
            if label[i] == 'NE':
                multi_label[i, 6] = 1   # Mark neutral if original label is 'NE'
            else:
                multi_label[i, 6] = 0
        return multi_label

In [ ]:
target = get_label(emotion, label, distribution, 0)
multi_label = get_label(emotion, label, distribution, 1)

In [ ]:
m_tr_data, m_te_data, w_tr_data, w_te_data, tr_label, te_label, tr_dis, te_dis = train_test_split(
    data, wire_data, target, distribution, test_size=0.2, random_state=42)

#### Save data partitioning

In [ ]:
directory = './data/3d_graph2/'
np.save(f'{directory}m_tr_data.npy', m_tr_data)
np.save(f'{directory}m_te_data.npy', m_te_data)
np.save(f'{directory}tr_label.npy', tr_label)
np.save(f'{directory}te_label.npy', te_label)
np.save(f'{directory}tr_dis.npy', tr_dis)
np.save(f'{directory}te_dis.npy', te_dis)

### FCCov-Net train

In [ ]:
dist_mode = 3 # Hellinger
device = 'cuda:2'
arch = 'DenseNet'   # LeNet
iter_num = 500
batch_size = 64
d = 32
bl = 2e-6

In [ ]:
# use our trained parameters
train_from_scratch = False

mccov_net = CustomNet(architecture=arch, optimizer_name='Adam', iter_num=iter_num, batch_size=batch_size, lr=1e-3,
                  blambda=bl, dist_mode=dist_mode, device=device, output_dim=d)

if train_from_scratch:
    mccov_net.train_step(m_tr_data, tr_dis, mccov_net.mccov_loss)
    # save model parameters
    torch.save(mccov_net.state_dict(), './data/3d_graph2/3d_dense_2e-6.pth')
else:
    mccov_net.load_state_dict(torch.load('./data/3d_graph2/3d_dense_2e-6.pth'))


In [ ]:
y_dcorfnorms = dcor.distance_correlation(mccov_net.transform(m_tr_data), tr_dis)
print(y_dcorfnorms)
y_dcorfnorms = dcor.distance_correlation(mccov_net.transform(m_te_data), te_dis)
print(y_dcorfnorms)

#### WIRE train

In [ ]:
from scipy.spatial.distance import pdist, squareform

def hellinger(p, q):
    return np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)) / np.sqrt(2)

def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

# Distance matrix required by WIRE
def distance_matrix(y, mode):
    if mode==0:   # Euclidean distance for Euclidean vectors
        diff = y[:, np.newaxis, :] - y[np.newaxis, :, :]
        D = np.sqrt(np.sum(diff**2, axis=-1))
    if mode==1:   # Spherical distance
        temp = y@(y.T)
        ma = np.maximum(-1, np.minimum(1, temp))
        D = np.arccos(ma)
        np.fill_diagonal(D,0)
    if mode==2:   # SPD manifold distance
        n = y.shape[0]
        p = y.shape[1]
        D = np.zeros((n,n))
        for i in range(n):
            for j in range(n):
                if i<j:
                    D[i,j] = spd_dis(y[i].reshape((1,p,p)),y[j].reshape((1,p,p))).item()
                D[j,i] = D[i,j]
    if mode==3:   # Hellinger distance
        pairwise_dists = pdist(y, metric=hellinger)
        D = squareform(pairwise_dists)
    if mode==4:    # Total Variation distance
        pairwise_dists = pdist(y, metric=tv_distance)
        D = squareform(pairwise_dists)
    return D

In [ ]:
# wire
D = distance_matrix(tr_dis, dist_mode) # Compute distance matrix
# eps = 1e-5
# gamma = 5
# Cross-validation for WIRE
cv_wire = True
eps, gamma = param(w_tr_data, D, cv_wire)

In [ ]:
train_res, test_res = wire(w_tr_data, w_te_data, D, d, eps, gamma)

In [ ]:
trd = dcor.distance_correlation(train_res, tr_dis.astype(np.float64))
ted = dcor.distance_correlation(test_res, te_dis.astype(np.float64))
print("Train dcor:", trd)
print("Test dcor:", ted)

### Draw

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import make_interp_spline

def plot_image_and_distribution_with_table(data, distribution, counts, save_path=None):
    fig = plt.figure(figsize=(15, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.6])

    # Plot image on the left
    ax0 = plt.subplot(gs[0])
    ax0.imshow(data, cmap='gray')
    ax0.axis('off')  # Hide the axes

    # Plot distribution in the middle
    ax1 = plt.subplot(gs[1])
    categories = ['HAP', 'SAD', 'SUR', 'ANG', 'DIS', 'FER']
    bars = ax1.bar(np.arange(len(distribution)), distribution, width=0.4)
    ax1.set_xticks(np.arange(len(categories)))
    ax1.set_xticklabels(categories)
    ax1.set_xlabel('Emotions')
    ax1.set_ylabel('Probabilities')
    ax1.set_title('Emotion Distribution')

    # Add smoothed dashed lines connecting the midpoints of the top of the bars
    midpoints = [bar.get_x() + bar.get_width() / 2 for bar in bars]
    heights = [bar.get_height() for bar in bars]

    # Interpolate for smooth curve
    x_new = np.linspace(min(midpoints), max(midpoints), 300)
    spline = make_interp_spline(midpoints, heights, k=2)  # k=2 for quadratic spline
    heights_smooth = spline(x_new)

    ax1.plot(x_new, heights_smooth, 'k--')

    # Create table on the right
    ax2 = plt.subplot(gs[2])
    ax2.axis('tight')
    ax2.axis('off')
    table_data = {
        'Emotion': ['HAP', 'SAD', 'SUR', 'ANG', 'DIS', 'FER'],
        'Score': counts
    }
    df = pd.DataFrame(table_data)
    table = ax2.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(15)
    table.scale(1.5, 3.3)

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.3)
    
    if save_path:
        plt.savefig(save_path)

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import make_interp_spline

def plot_image_and_distribution(data, distribution, save_path=None):
    fig = plt.figure(figsize=(10, 5))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1]) 

    # Plot image on the left
    ax0 = plt.subplot(gs[0])
    ax0.imshow(data, cmap='gray')
    ax0.axis('off')  # Hide the axes

    # Plot distribution in the middle
    ax1 = plt.subplot(gs[1])
    categories = ['HAP', 'SAD', 'SUR', 'ANG', 'DIS', 'FER']
    bars = ax1.bar(np.arange(len(distribution)), distribution, width=0.4)
    ax1.set_xticks(np.arange(len(categories)))
    ax1.set_xticklabels(categories)
    # ax1.set_xlabel('Emotions')
    # ax1.set_ylabel('Probabilities')
    # ax1.set_title('Emotion Distribution')

    # Add smoothed dashed lines connecting the midpoints of the top of the bars
    midpoints = [bar.get_x() + bar.get_width() / 2 for bar in bars]
    heights = [bar.get_height() for bar in bars]

    # Interpolate for smooth curve
    x_new = np.linspace(min(midpoints), max(midpoints), 300)
    spline = make_interp_spline(midpoints, heights, k=2)  # k=2 for quadratic spline
    heights_smooth = spline(x_new)

    ax1.plot(x_new, heights_smooth, 'k--')

    plt.tight_layout()
    # plt.subplots_adjust(wspace=0.3) 
    
    if save_path:
        plt.savefig(save_path)

    plt.show()

In [ ]:
j = 1
plot_image_and_distribution_with_table(data[j], distribution[j], counts[j],'YM_sad')

In [ ]:
j = 42
plot_image_and_distribution(data[j], distribution[j],'tmp_4')

In [ ]:
def find_seven(arr, seven=6):
    sorted_indices = np.argsort(arr)

    if len(arr) >= seven:
        seventh_smallest_index = sorted_indices[seven-1]
        seventh_smallest_value = arr[seventh_smallest_index]
    else:
        seventh_smallest_index = None
        seventh_smallest_value = None
        
    return ( seventh_smallest_value, seventh_smallest_index)

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import os

anti_emotion = {0:'HA', 1:'SA', 2:'SU', 3:'AN', 4:'DI', 5:'FE', 6:'NE'}

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
ax.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
markers = ['o', 's', '^', 'D', 'v', 'P', '*']

temp = np.squeeze(m_tr_data, axis=1)
x = []
y = []
images = []

for i in range(7):
    mask = y_train == i
    img = temp[mask,:,:]
    xp = x_train[mask, 0]
    yp = x_train[mask, 1]
    ax.scatter(xp, yp, color=colors[i], marker=markers[i], 
                label=f'Class:{anti_emotion[i]}', alpha=0.7)
    
    if i==5:
        value, idx = find_seven(xp, 10)
    elif i==4:
        value, idx = find_seven(xp, 7)
    else:
        value, idx = find_seven(xp, 6)
    x.append(value)
    y.append(yp[idx])
    images.append(img[idx])
    
zoom = 0.4  
for i, (x0,y0) in enumerate(zip(x,y)):
    im = OffsetImage(images[i], zoom=zoom, cmap='gray')
    ab = AnnotationBbox(im, (x0+0.5, y0+0.8), frameon=False)
    ax.add_artist(ab)

plt.legend()
os.makedirs('./figures', exist_ok=True)
plt.savefig('./figures/2d_feature.png')
plt.show()

#### 3D Visualization

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# data_load
pos = './data/3d_graph2/'
vis_tr_data = np.load(f"{pos}m_tr_data.npy")
vis_tr_label = np.load(f"{pos}tr_label.npy")

In [ ]:
# model load
dist_mode = 3 # Hellinger
device = 'cuda:2'
arch = 'DenseNet'   # LeNet
iter_num = 500
batch_size = 64
d = 3
bl = 2e-6

vis_model = CustomNet(architecture=arch, optimizer_name='Adam', iter_num=iter_num, batch_size=batch_size, lr=1e-3,
                  blambda=bl, dist_mode=dist_mode, device=device, output_dim=d)

vis_model.load_state_dict(torch.load('./data/3d_graph2/3d_dense_2e-6.pth'))

x_train = vis_model.transform(vis_tr_data)
y_train = vis_tr_label

In [ ]:
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
markers = ['o', 's', '^', 'D', 'v', 'P', '*']
anti_emotion = {0:'HA', 1:'SA', 2:'SU', 3:'AN', 4:'DI', 5:'FE', 6:'NE'}

# Create a 3D figure object
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
# ax.set_axis_off()

# Plot scatter points according to class labels in y_train
for i in range(7):  # assuming 7 classes
    indices = y_train == i
    ax.scatter(x_train[indices, 0], x_train[indices, 1], x_train[indices, 2], 
               c=colors[i], marker=markers[i], label=f'Class {anti_emotion[i]}', s=20, alpha=0.6)
# Adjust viewing angle
ax.view_init(elev=5, azim=0)


# Display legend
ax.legend()
# plt.savefig('./figures/3d_angle1')
# Show the plot
plt.show()

In [ ]:
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
markers = ['o', 's', '^', 'D', 'v', 'P', '*']
anti_emotion = {0:'HA', 1:'SA', 2:'SU', 3:'AN', 4:'DI', 5:'FE', 6:'NE'}

# Create a 3D figure object
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
# ax.set_axis_off()

# Plot scatter points according to class labels in y_train
for i in range(7):  # assuming 7 classes
    indices = y_train == i
    ax.scatter(x_train[indices, 0], x_train[indices, 1], x_train[indices, 2], 
               c=colors[i], marker=markers[i], label=f'Class {anti_emotion[i]}', s=20, alpha=0.6)
# Change viewing angle
ax.view_init(elev=5, azim=90)

# Display legend
ax.legend()
# plt.savefig('./figures/3d_angle2')
# Show the plot
plt.show()

In [ ]:
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
markers = ['o', 's', '^', 'D', 'v', 'P', '*']
anti_emotion = {0:'HA', 1:'SA', 2:'SU', 3:'AN', 4:'DI', 5:'FE', 6:'NE'}

# Create a 3D figure object
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
# ax.set_axis_off()

# Plot scatter points according to class labels in y_train
for i in range(7):  # assuming 7 classes
    indices = y_train == i
    ax.scatter(x_train[indices, 0], x_train[indices, 1], x_train[indices, 2], 
               c=colors[i], marker=markers[i], label=f'Class {anti_emotion[i]}', s=20, alpha=0.6)
# Change viewing angle
ax.view_init(elev=90, azim=5)
    
# Display legend
ax.legend()
# plt.savefig('./figures/3d_angle3')
# Show the plot
plt.show()